In [1]:
import pandas as pd
import random 
from itertools import combinations
import math

In [4]:
class ts():
    def __init__(self,path,seed,tabu_tenure):
        self.path=path
        self.seed=seed
        self.tabu_tenure=tabu_tenure
        self.instance_dict=self.input_data()
        self.initial_sol=self.get_initial_sol()
        self.tabu_str,self.best_sol,self.best_obj=self.TSearch()

    def input_data(self):
        return pd.read_excel(self.path,names=["job","weight","proccesing_time","due_date"],index_col=0).to_dict("index")

    def get_tabustructure(self):
        dict={}
        for i in combinations(self.instance_dict.keys(),2):
            dict[i]={"tabu_time":0,"movevalue":0}

        return dict

    def get_initial_sol(self):
        n_jobs=len(self.instance_dict)
        initial_sol=list(range(1,n_jobs+1))
        random.seed(self.seed)
        random.shuffle(initial_sol)
        return initial_sol

    def obj(self,solution):
        dict=self.instance_dict
        t=0
        obj_val=0
        for i in solution:
            ci=t+dict[i]["proccesing_time"]
            di=dict[i]["due_date"]
            ti=max(0,ci-di)
            wi=dict[i]["weight"]
            obj_val+=wi*ti
            t=ci
        return obj_val

    def swapmove(self,solution,i,j):
        solution=solution.copy()
        i_index=solution.index(i)
        j_index=solution.index(j)
        solution[i_index],solution[j_index]=solution[j_index],solution[i_index]
        return solution

    def TSearch(self):
        tenure=self.tabu_tenure
        tabu_structure=self.get_tabustructure()
        best_sol=self.initial_sol
        best_obj=self.obj(best_sol)
        current_sol=self.initial_sol
        current_obj=self.obj(current_sol)
        iter=0
        terminate=0
        while terminate<100:
            if iter<=10:
                print(f"iter-{iter}-best_obj-{best_obj}")
            for move in tabu_structure:
                cand_sol=self.swapmove(current_sol,move[0],move[1])
                cand_obj=self.obj(cand_sol)
                tabu_structure[move]["moveval"]=cand_obj

            while 1:
                best_move=min(tabu_structure,key=lambda x:tabu_structure[x]["moveval"])
                moveval=tabu_structure[best_move]["moveval"]
                tabu_time=tabu_structure[best_move]["tabu_time"]

                if tabu_time<iter:
                    current_sol=self.swapmove(current_sol,best_move[0],best_move[1])
                    current_obj=self.obj(current_sol)

                    if moveval<best_obj:
                        best_sol=current_sol
                        best_obj=current_obj
                        terminate=0
                    else:
                        terminate+=1
                    tabu_structure[best_move]["tabu_time"]=iter+tenure

                    iter+=1
                    break
                else:
                    if moveval<best_obj:
                        current_sol=self.swapmove(current_sol,best_move[0],best_move[1])
                        current_obj=self.obj(current_sol)
                        best_sol=current_sol
                        best_obj=current_obj
                        terminate=0
                        iter+=1
                        break
                    else:
                        tabu_structure[best_move]["moveval"]=float("inf")
                        continue

        print("completedddd")
        print(f"performed iter-{iter},bes sol-{best_sol},best_obj-{best_obj}",sep="\n")

        return tabu_structure,best_sol,best_obj

In [5]:
print("starteddd")
test=ts(path="Instance_10.xlsx",seed=2012,tabu_tenure=3)

starteddd
iter-0-best_obj-39.080000000000005
iter-1-best_obj-26.13
iter-2-best_obj-17.930000000000003
iter-3-best_obj-14.960000000000003
iter-4-best_obj-14.200000000000003
iter-5-best_obj-13.830000000000002
iter-6-best_obj-13.530000000000001
iter-7-best_obj-13.370000000000001
iter-8-best_obj-13.260000000000002
iter-9-best_obj-13.240000000000002
iter-10-best_obj-13.240000000000002
completedddd
performed iter-109,bes sol-[3, 2, 1, 4, 8, 10, 5, 9, 7, 6],best_obj-13.240000000000002
